# Simple lead-time tests for FINE

This notebook is intentionally close to the small FINE examples:

1. import `fine` and `pandas`
2. create a one-node `EnergySystemModel`
3. add one `Source` and one `Sink`
4. optimize
5. inspect `cap`, `commis`, and `decommis`

The tests isolate the new lead-time logic. CAPEX timing is **not** tested here.

Assumed implementation logic:

```text
availableCommis[ip] = commis[ip - roundedLeadTime]
cap[ip] = cap[ip-1] + availableCommis[ip] - decommis[ip]
decommis[ip] = commis[ip - roundedLeadTime - roundedLifetime]
```

`commis[ip]` is treated as the investment/commissioning decision. It becomes physically available only after `leadTime`.


Note: FINE input dictionaries use investment-period names (`2020`, `2025`, ...). The helper functions convert the test dictionaries from internal IP indices to those year keys.

## 1. Import packages

Change `SOLVER` if your local FINE setup uses another solver.

In [1]:
import inspect

import fine as fn
import pandas as pd
import pyomo.environ as pyomo

SOLVER = "gurobi"  # change to "glpk", "cbc", "highs", ... if needed

assert "leadTime" in inspect.signature(fn.Source.__init__).parameters, (
    "Source.__init__ does not accept leadTime yet. "
    "Add leadTime to Source, Sink, Conversion, Storage, Transmission, and Component first."
)

print("FINE imported.")
print("Using solver:", SOLVER)
print("Source accepts leadTime:", "leadTime" in inspect.signature(fn.Source.__init__).parameters)


FINE imported.
Using solver: gurobi
Source accepts leadTime: True


## 2. Helper functions

The helper builds a very small one-node electricity model. The source capacity is fixed through `commissioningFix`, so the tests focus only on whether `cap` appears at the correct investment period.

Important FINE convention: time-dependent input dictionaries use the **investment-period names** as keys, e.g. `2020`, `2025`, not the internal indices `0`, `1`. The test calls below still use internal IP indices for readability; the helper converts them to years before passing them to FINE.

In [2]:
LOCATION = "node1"
COMMODITY = "electricity"
SOURCE_NAME = "lead_time_source"
SINK_NAME = "electricity_demand"
START_YEAR = 2020


def ts(value):
    """One-step time series for one location."""
    return pd.DataFrame([value], index=[0], columns=[LOCATION])


def loc_series(value):
    """One-location Series."""
    return pd.Series([value], index=[LOCATION])


def ip_year(ip, interval, start_year=START_YEAR):
    """Convert internal FINE investment-period index to investment-period name/year."""
    return start_year + ip * interval


def ip_time_series(values_by_ip, n_ips, interval):
    """Dictionary of one-step time series keyed by FINE investment-period names.

    The function accepts values_by_ip keyed by internal IP indices for easier tests,
    but returns a dictionary keyed by years, as required by FINE input processing.
    """
    return {
        ip_year(ip, interval): ts(values_by_ip.get(ip, 0))
        for ip in range(n_ips)
    }


def ip_series(values_by_ip, n_ips, interval):
    """Dictionary of one-location Series keyed by FINE investment-period names.

    The function accepts values_by_ip keyed by internal IP indices for easier tests,
    but returns a dictionary keyed by years, as required by FINE input processing.
    """
    return {
        ip_year(ip, interval): loc_series(values_by_ip.get(ip, 0))
        for ip in range(n_ips)
    }


def build_model(
    *,
    n_ips,
    interval,
    lead_time,
    technical_lifetime,
    demand_by_ip,
    commissioning_by_ip,
):
    """Build a minimal one-node FINE model for testing lead-time capacity logic."""
    esM = fn.EnergySystemModel(
        locations={LOCATION},
        commodities={COMMODITY},
        commodityUnitsDict={COMMODITY: "MW_el"},
        numberOfTimeSteps=1,
        hoursPerTimeStep=1,
        startYear=START_YEAR,
        numberOfInvestmentPeriods=n_ips,
        investmentPeriodInterval=interval,
        costUnit="Euro",
        lengthUnit="km",
        verboseLogLevel=0,
    )

    esM.add(
        fn.Source(
            esM=esM,
            name=SOURCE_NAME,
            commodity=COMMODITY,
            hasCapacityVariable=True,
            operationRateMax=ip_time_series({ip: 1 for ip in range(n_ips)}, n_ips, interval),
            commissioningFix=ip_series(commissioning_by_ip, n_ips, interval),
            investPerCapacity=1,
            economicLifetime=5,
            technicalLifetime=technical_lifetime,
            leadTime=lead_time,
        )
    )

    esM.add(
        fn.Sink(
            esM=esM,
            name=SINK_NAME,
            commodity=COMMODITY,
            hasCapacityVariable=False,
            operationRateFix=ip_time_series(demand_by_ip, n_ips, interval),
        )
    )

    return esM


def optimize(esM):
    esM.optimize(timeSeriesAggregation=False, solver=SOLVER)


def read_design_variables(esM, component=SOURCE_NAME, location=LOCATION):
    """Read cap, commis, and decommis directly from Pyomo variables."""
    abbrv = esM.componentModelingDict["SourceSinkModel"].abbrvName
    rows = []

    for ip in esM.investmentPeriods:
        row = {"ip": ip, "year": esM.investmentPeriodNames[ip]}
        for var_name in ["cap", "commis", "decommis"]:
            var = getattr(esM.pyM, f"{var_name}_{abbrv}")
            try:
                row[var_name] = pyomo.value(var[location, component, ip])
            except KeyError:
                row[var_name] = 0.0
        rows.append(row)

    return pd.DataFrame(rows).set_index("ip")


def assert_expected(results, column, expected, tol=1e-6):
    for ip, expected_value in expected.items():
        actual_value = results.loc[ip, column]
        assert abs(actual_value - expected_value) <= tol, (
            f"Expected {column}[{ip}] = {expected_value}, got {actual_value}"
        )


## Test 1 — `leadTime = 0`: original immediate availability

The source is forced to decide/build 10 MW in IP 0.
With `leadTime = 0`, this capacity must be available immediately in IP 0.

Expected:

| IP | commis | cap | decommis |
|---:|---:|---:|---:|
| 0 | 10 | 10 | 0 |
| 1 | 0 | 10 | 0 |


In [3]:
esM = build_model(
    n_ips=2,
    interval=5,
    lead_time=0,
    technical_lifetime=20,
    demand_by_ip={0: 10, 1: 10},
    commissioning_by_ip={0: 10, 1: 0},
)

optimize(esM)
results = read_design_variables(esM)
display(results)

assert_expected(results, "commis", {0: 10, 1: 0})
assert_expected(results, "cap", {0: 10, 1: 10})
assert_expected(results, "decommis", {0: 0, 1: 0})

print("Test 1 passed: leadTime=0 preserves immediate availability.")


Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 10 rows, 12 columns and 21 nonzeros (Min)
Model fingerprint: 0xa39f9077
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [7e-01, 1e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 10 rows and 12 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0800000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal obj

,year,cap,commis,decommis
ip,,,,
0,2020,10.0,10.0,0.0
1,2025,10.0,0.0,0.0


Test 1 passed: leadTime=0 preserves immediate availability.


## Test 2 — `leadTime = 5`, interval 5: delayed availability by one IP

The same 10 MW decision is forced in IP 0.
With `leadTime = 5` and `investmentPeriodInterval = 5`, the capacity must **not** be available in IP 0. It becomes available in IP 1.

This test checks the modified `initialYear` and `designDevelopmentConstraint` logic.

Expected:

| IP | commis | cap | decommis |
|---:|---:|---:|---:|
| 0 | 10 | 0 | 0 |
| 1 | 0 | 10 | 0 |


In [4]:
esM = build_model(
    n_ips=2,
    interval=5,
    lead_time=5,
    technical_lifetime=20,
    demand_by_ip={0: 0, 1: 10},
    commissioning_by_ip={0: 10, 1: 0},
)

optimize(esM)
results = read_design_variables(esM)
display(results)

assert_expected(results, "commis", {0: 10, 1: 0})
assert_expected(results, "cap", {0: 0, 1: 10})
assert_expected(results, "decommis", {0: 0, 1: 0})

print("Test 2 passed: leadTime=5 delays availability by one investment period.")


Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 10 rows, 12 columns and 20 nonzeros (Min)
Model fingerprint: 0xc2c7b99a
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [7e-01, 1e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 10 rows and 12 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0800000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal obj

,year,cap,commis,decommis
ip,,,,
0,2020,0.0,10.0,0.0
1,2025,10.0,0.0,0.0


Test 2 passed: leadTime=5 delays availability by one investment period.


## Test 3 — lead time plus technical lifetime: delayed decommissioning

Now the model has four investment periods.

Settings:

```text
investmentPeriodInterval = 5 years
leadTime = 5 years      -> rounded lead = 1 IP
technicalLifetime = 10  -> rounded lifetime = 2 IPs
```

The source is forced to decide/build 10 MW in IP 0.
It becomes available in IP 1 and should be decommissioned in IP 3.

Expected:

| IP | commis | cap | decommis |
|---:|---:|---:|---:|
| 0 | 10 | 0 | 0 |
| 1 | 0 | 10 | 0 |
| 2 | 0 | 10 | 0 |
| 3 | 0 | 0 | 10 |


In [5]:
esM = build_model(
    n_ips=4,
    interval=5,
    lead_time=5,
    technical_lifetime=10,
    demand_by_ip={0: 0, 1: 10, 2: 10, 3: 0},
    commissioning_by_ip={0: 10, 1: 0, 2: 0, 3: 0},
)

optimize(esM)
results = read_design_variables(esM)
display(results)

assert_expected(results, "commis", {0: 10, 1: 0, 2: 0, 3: 0})
assert_expected(results, "cap", {0: 0, 1: 10, 2: 10, 3: 0})
assert_expected(results, "decommis", {0: 0, 1: 0, 2: 0, 3: 10})

print("Test 3 passed: decommissioning happens after lead time plus technical lifetime.")


Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 20 rows, 24 columns and 43 nonzeros (Min)
Model fingerprint: 0x45cc0e82
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e-01, 1e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 20 rows and 24 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0800000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal obj

,year,cap,commis,decommis
ip,,,,
0,2020,0.0,10.0,0.0
1,2025,10.0,0.0,0.0
2,2030,10.0,0.0,0.0
3,2035,0.0,0.0,10.0


Test 3 passed: decommissioning happens after lead time plus technical lifetime.


## Test 4 — lead time shorter than interval: `ceil` rounding

This test checks the chosen rounding rule for lead times.

```text
investmentPeriodInterval = 5 years
leadTime = 2 years
ipLeadTime = 2 / 5 = 0.4
roundedIpLeadTime = ceil(0.4) = 1
```

So even a 2-year lead time delays availability to the next modeled investment period.

Expected:

| IP | commis | cap |
|---:|---:|---:|
| 0 | 10 | 0 |
| 1 | 0 | 10 |


In [6]:
esM = build_model(
    n_ips=2,
    interval=5,
    lead_time=2,
    technical_lifetime=20,
    demand_by_ip={0: 0, 1: 10},
    commissioning_by_ip={0: 10, 1: 0},
)

optimize(esM)
results = read_design_variables(esM)
display(results)

assert_expected(results, "commis", {0: 10, 1: 0})
assert_expected(results, "cap", {0: 0, 1: 10})
assert_expected(results, "decommis", {0: 0, 1: 0})

print("Test 4 passed: positive leadTime shorter than the interval is rounded up to one IP.")


Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 10 rows, 12 columns and 20 nonzeros (Min)
Model fingerprint: 0xc2c7b99a
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [7e-01, 1e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 10 rows and 12 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0800000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal obj

,year,cap,commis,decommis
ip,,,,
0,2020,0.0,10.0,0.0
1,2025,10.0,0.0,0.0


Test 4 passed: positive leadTime shorter than the interval is rounded up to one IP.


## What each test checks

- **Test 1**: `leadTime=0` keeps original FINE behavior.
- **Test 2**: `commis[0]` does not enter `cap[0]` when `leadTime>0`; it enters `cap[1]`.
- **Test 3**: decommissioning is shifted by `leadTime + technicalLifetime`.
- **Test 4**: the `ceil` rounding of positive lead times works.

If one test fails, the likely source is:

- `cap[0]` wrong → check `stockCapacityConstraint` / `initialYear`.
- `cap[1]` wrong → check `designDevelopmentConstraint`.
- `decommis[3]` wrong → check `decommissioningConstraint`.
- Test 4 wrong → check `self.roundedIpLeadTime = self.ipLeadTime.apply(math.ceil)`.
